# Cadastral AI Mapper — GIS & ML Exploration Notebook
**Smart India Hackathon 2026**

This notebook demonstrates the end-to-end workflow:
1. Loading sample Cadastral GeoJSON data.
2. Generating 14-digit national ULPIN codes.
3. Running Shapely-based geometric topology conflict detection.
4. Segmenting candidate building footprints using AI/CV pipelines.

In [ ]:
import json
from pathlib import Path
from shapely.geometry import shape, Polygon, mapping
import geopandas as gpd
import matplotlib.pyplot as plt

# Set base paths
DATA_PATH = Path("../data/sample_area/sample_parcels.geojson")
with open(DATA_PATH, "r", encoding="utf-8") as f:
    sample_data = json.load(f)

print(f"Loaded {len(sample_data['features'])} sample parcels.")

## 1. ULPIN (Unique Land Parcel Identification Number) Generation

In [ ]:
from ml_pipeline.id_generator import generate_ulpin, parse_ulpin

# Sample coordinate in Bengaluru Urban (State 29, District 572)
lat, lon = 12.93510, 77.62010
ulpin = generate_ulpin(lat, lon, state_code="29", district_code="572")
print("Generated ULPIN:", ulpin)
print("Parsed Structure:", parse_ulpin(ulpin))

## 2. Geometric Topology Conflict Matrix Detection

In [ ]:
from ml_pipeline.geometry import detect_topology_conflicts, calculate_metric_metrics

conflicts = detect_topology_conflicts(sample_data["features"], overlap_threshold_sqm=1.0)
print(f"Detected {len(conflicts)} active topology conflict(s):")
for c in conflicts:
    print(f" - Conflict: {c['conflict_type']} | Severity: {c['severity']} | Overlap Area: {c['overlap_area_sqm']} m²")

## 3. Visualizing Parcel Footprints with GeoPandas

In [ ]:
# Convert GeoJSON to GeoDataFrame for spatial plotting
gdf = gpd.read_file(str(DATA_PATH))
fig, ax = plt.subplots(figsize=(10, 8))
gdf.plot(ax=ax, column="status", cmap="viridis", legend=True, alpha=0.6, edgecolor="black")
plt.title("Sample Cadastral Parcels by Status (Koramangala, Bengaluru)")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.show()